<a href="https://colab.research.google.com/github/Takumi173/Test/blob/main/Dataset_JSON_Reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 準備

## 処理用にデータを結合

In [1]:
# データのコピー
!git clone https://github.com/cdisc-org/sdtm-adam-pilot-project.git

Cloning into 'sdtm-adam-pilot-project'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 224 (delta 64), reused 220 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 24.51 MiB | 3.71 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (87/87), done.


In [2]:
# 使用するjsonデータとdefine.xmlを新規ディレクトリにコピーする

import os
import shutil
import json

source_dir = "sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm"
json_dir   = "json_files"
define_dir = "define_xml"

if not os.path.exists(json_dir):
    os.makedirs(json_dir)

if not os.path.exists(define_dir):
    os.makedirs(define_dir)

for root, _, files in os.walk(source_dir):
  for file in files:
    if file.endswith(".json"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(json_dir, file)
      shutil.copy(source_path, target_path)
    if file.endswith("define.xml"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(define_dir, file)
      shutil.copy(source_path, target_path)

In [3]:
# jsonファイルをリスト形式に結合したファイル（dataset_list.json）を作成

dataset_list = []
for filename in os.listdir(json_dir):
  if filename.endswith(".json"):
    with open(os.path.join(json_dir, filename), "r") as f:
      try:
        json_data = json.load(f)
        dataset_list.append(json_data)
      except json.JSONDecodeError as e:
        print(f"Error decoding JSON in file {filename}: {e}")

with open("dataset_list.json", "w") as f:
  json.dump(dataset_list, f)


## 症例フィルタリング関数の定義

In [4]:
def filter_data(data, target_usubjids):
    """
    複数のドメインデータを含むリストから、指定されたUSUBJIDのrowsのみを抽出して新しいJSONファイルに保存する。
    入力データがリストでない場合はエラーメッセージを出力する。
    データ構造は、"columns" 内の "name" が "USUBJID" の列を持つことを前提とする。

    Args:
        data (list): ドメインを結合させたのリスト。リストでない場合はエラーとなる。
        output_file (str): 出力するJSONファイル名。
        target_usubjids (list): 残したいUSUBJIDのリスト。
    """
    if not isinstance(data, list):
        print("エラー：入力データはJSONオブジェクトのリストである必要があります。")
        return

    filtered_data_list = []
    for item in data:
        usubjid_index = -1
        if 'columns' in item:
            for i, col in enumerate(item['columns']):
                if 'name' in col and col['name'] == 'USUBJID':
                    usubjid_index = i
                    break

        if usubjid_index == -1:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'name' が 'USUBJID' の列が見つかりません。スキップします。")
            filtered_data_list.append(item)
            continue

        if 'rows' in item:
            filtered_rows = [
                row for row in item['rows'] if len(row) > usubjid_index and row[usubjid_index] in target_usubjids
            ]
            new_data = item.copy()
            new_data['rows'] = filtered_rows
            new_data['records'] = len(filtered_rows)
            filtered_data_list.append(new_data)
        else:
            print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'rows' が見つかりません。スキップします。")
            filtered_data_list.append(item)

    return filtered_data_list



with open('dataset_list.json', 'r') as f:
    data = json.load(f)

target_ids = ['01-701-1211']
output_filename = 'filtered_list.json'

filtered_data_list = filter_data(data, target_ids)

with open(output_filename, 'w') as f:
    json.dump(filtered_data_list, f)
print(f"処理完了：'{output_filename}' に USUBJID が {target_ids} のデータを出力しました。")

警告：データセット 'CDISCPILOT01.ta' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.te' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ti' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.ts' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
警告：データセット 'CDISCPILOT01.tv' に 'name' が 'USUBJID' の列が見つかりません。スキップします。
処理完了：'filtered_list.json' に USUBJID が ['01-701-1211'] のデータを出力しました。


## データ書き換え関数の定義

In [5]:
def data_update(data, target_domain, target_usubjid, target_seq, target_variable, new_value):
    """
    指定されたUSUBJIDを持つレコードの指定された変数を書き換えます。
    {target_domain}SEQが存在する場合はそれもキーとして使用します。
    元のデータは変更せず、新しいデータ構造を返します。

    Args:
        data (list): データ全体のリスト。指定されたtarget_domainのデータセットを含むことを想定します。
        target_domain (str): 対象のドメイン名（例: "CM"）。
        target_usubjid (str): 書き換えたいレコードのUSUBJID。
        target_seq (int): 書き換えたいレコードの{target_domain}SEQの値（存在しない場合は無視されます）。
        target_variable (str): 書き換えたい変数の名前（例: "CMTRT"）。
        new_value (any): 新しい変数の値。

    Returns:
        list: 指定された変数が更新された新しいデータ全体のリスト。
              該当するレコードが見つからなかった場合、元のデータのコピーを返します。
    """
    updated_data = []
    seqname = target_domain + 'SEQ'

    for dataset in data:
        updated_dataset = dataset.copy()
        if updated_dataset.get("itemGroupOID") == target_domain:
            updated_rows = []
            found = False
            usubjid_index = -1
            seq_index = -1
            variable_index = -1
            has_seq = False

            for i, col in enumerate(updated_dataset["columns"]):
                if col["name"] == "USUBJID":
                    usubjid_index = i
                elif col["name"] == seqname:
                    seq_index = i
                    has_seq = True
                elif col["name"] == target_variable:
                    variable_index = i

            if usubjid_index != -1 and variable_index != -1:
                for row in dataset["rows"]:
                    updated_row = list(row)  # 行をコピーして変更
                    usubjid_match = updated_row[usubjid_index] == target_usubjid
                    seq_match = True
                    if has_seq and seq_index != -1:
                        seq_match = (len(updated_row) > seq_index and updated_row[seq_index] == target_seq)
                    elif has_seq:
                        print(f"警告: '{target_domain}' データセットに '{seqname}' 列が見つかりましたが、インデックスが無効です。USUBJIDのみをキーとして使用します。")

                    if usubjid_match and seq_match:
                        updated_row[variable_index] = new_value
                        if has_seq and seq_index != -1:
                            print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' の '{target_variable}' を '{new_value}' に更新しました。")
                        else:
                            print(f"USUBJID '{target_usubjid}' の '{target_variable}' を '{new_value}' に更新しました。")
                        found = True
                    updated_rows.append(updated_row)
                updated_dataset["rows"] = updated_rows
            elif updated_dataset.get("itemGroupOID") == target_domain:
                print(f"'{target_domain}' データセットに 'USUBJID' または '{target_variable}' 列が見つかりませんでした。")

            updated_data.append(updated_dataset)
            if not found and updated_dataset.get("itemGroupOID") == target_domain:
                if has_seq and seq_index != -1:
                    print(f"USUBJID '{target_usubjid}'、'{seqname}' '{target_seq}' に該当するレコードが見つかりませんでした。")
                else:
                    print(f"USUBJID '{target_usubjid}' に該当するレコードが見つかりませんでした。")
        else:
            updated_data.append(updated_dataset)

    if not any(d.get("itemGroupOID") == target_domain for d in data):
        print(f"{target_domain} データセットが見つかりませんでした。")

    return updated_data

# 例：書き換えテスト
updated_data = data_update(filtered_data_list, "DM", "01-701-1211", 0, "AGE", 49)
updated_data = data_update(updated_data, "CM", "01-701-1211", 3, "CMTRT", "New Drug 123456789")
updated_data = data_update(updated_data, "CM", "01-701-1211", 0, "CMDOSE", 123)

USUBJID '01-701-1211' の 'AGE' を '49' に更新しました。
USUBJID '01-701-1211'、'CMSEQ' '3' の 'CMTRT' を 'New Drug 123456789' に更新しました。
USUBJID '01-701-1211'、'CMSEQ' '0' に該当するレコードが見つかりませんでした。


## データ比較関数の定義

In [6]:
from typing import List, Dict, Any

def compare_data(old_data: List[Dict[str, Any]], new_data: List[Dict[str, Any]]) -> None:
    """
    2つのデータリストの更新差分を人間が読みやすい形式で出力します。

    Args:
        old_data: 旧データリスト。
        new_data: 新データリスト。
    """

    def create_row_dict(item_group: Dict[str, Any], row: List[Any]) -> Dict[str, Any]:
        """rowデータをキー付きの辞書に変換する"""
        row_dict = {}
        for i, column in enumerate(item_group['columns']):
            row_dict[column['name']] = row[i]
        return row_dict

    def get_key_values(item_group_oid: str, row_dict: Dict[str, Any]) -> Dict[str, Any]:
        """データのキーとなる値を抽出する"""
        key_values = {'USUBJID': row_dict.get('USUBJID')}
        seq_key = f"{item_group_oid}SEQ"
        if seq_key in row_dict:
            key_values[seq_key] = row_dict[seq_key]
        return key_values

    def format_key(key_values: Dict[str, Any]) -> str:
        """キー値を人間が読みやすい文字列に整形する"""
        parts = []
        for key, value in key_values.items():
            if value is not None:
                parts.append(f"{key} = {value}")
        return ", ".join(parts)

    old_data_by_group = {item['itemGroupOID']: item for item in old_data}
    new_data_by_group = {item['itemGroupOID']: item for item in new_data}

    all_group_oids = set(old_data_by_group.keys()) | set(new_data_by_group.keys())

    for group_oid in sorted(list(all_group_oids)):
        print(f"--- ItemGroupOID: {group_oid} ---")
        old_group = old_data_by_group.get(group_oid)
        new_group = new_data_by_group.get(group_oid)

        old_rows_by_key = {}
        if old_group and 'rows' in old_group:
            for row in old_group['rows']:
                row_dict = create_row_dict(old_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    old_rows_by_key[format_key(key_values)] = row_dict

        new_rows_by_key = {}
        if new_group and 'rows' in new_group:
            for row in new_group['rows']:
                row_dict = create_row_dict(new_group, row)
                if 'USUBJID' in row_dict and row_dict['USUBJID'] is not None:
                    key_values = get_key_values(group_oid, row_dict)
                    new_rows_by_key[format_key(key_values)] = row_dict

        old_keys = set(old_rows_by_key.keys())
        new_keys = set(new_rows_by_key.keys())

        # 追加されたデータ
        added_keys = new_keys - old_keys
        for key in sorted(list(added_keys)):
            print(f"{key}:")
            print("  Added")
            for item_key, old_value in sorted(new_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 削除されたデータ
        removed_keys = old_keys - new_keys
        for key in sorted(list(removed_keys)):
            print(f"{key}:")
            print("  Deleted")
            for item_key, old_value in sorted(old_rows_by_key[key].items()):
                print(f"    {item_key}: {old_value}")
            print()

        # 更新されたデータ
        common_keys = old_keys & new_keys
        for key in sorted(list(common_keys)):
            if old_rows_by_key[key] != new_rows_by_key[key]:
                print(f"{key}:")
                print("  Updated:")
                old_row = old_rows_by_key[key]
                new_row = new_rows_by_key[key]
                for item_key in sorted(list(set(old_row.keys()) | set(new_row.keys()))):
                    old_value = old_row.get(item_key)
                    new_value = new_row.get(item_key)
                    if old_value != new_value:
                        print(f"    {item_key}: {old_value!r} -> {new_value!r}")
                print()


# 比較実行
compare_data(filtered_data_list, updated_data)

--- ItemGroupOID: AE ---
--- ItemGroupOID: CM ---
USUBJID = 01-701-1211, CMSEQ = 3:
  Updated:
    CMTRT: 'HYDROCORTISONE' -> 'New Drug 123456789'

--- ItemGroupOID: DM ---
USUBJID = 01-701-1211:
  Updated:
    AGE: 76 -> 49

--- ItemGroupOID: DS ---
--- ItemGroupOID: EX ---
--- ItemGroupOID: LB ---
--- ItemGroupOID: MH ---
--- ItemGroupOID: QS ---
--- ItemGroupOID: RELREC ---
--- ItemGroupOID: SC ---
--- ItemGroupOID: SE ---
--- ItemGroupOID: SUPPAE ---
--- ItemGroupOID: SUPPDM ---
--- ItemGroupOID: SUPPDS ---
--- ItemGroupOID: SUPPLB ---
--- ItemGroupOID: SV ---
--- ItemGroupOID: TA ---
--- ItemGroupOID: TE ---
--- ItemGroupOID: TI ---
--- ItemGroupOID: TS ---
--- ItemGroupOID: TV ---
--- ItemGroupOID: VS ---


In [7]:
import json

with open('/content/dataset_list.json', 'r') as f:
    data = json.load(f)

usubjids = set()
for dataset in data:
    if 'columns' in dataset:
        for i, col in enumerate(dataset['columns']):
            if 'name' in col and col['name'] == 'USUBJID':
                if 'rows' in dataset:
                    for row in dataset['rows']:
                        if len(row) > i:
                            usubjids.add(row[i])

print(list(usubjids))


['01-704-1017', '01-703-1076', '01-708-1178', '01-716-1229', '01-704-1325', '01-701-1307', '01-705-1280', '01-705-1112', '01-716-1441', '01-708-1067', '01-710-1077', '01-703-1096', '01-706-1049', '01-701-1239', '01-705-1059', '01-714-1035', '01-717-1357', '01-708-1352', '01-709-1001', '01-710-1235', '01-709-1238', '01-716-1063', '01-704-1218', '01-701-1180', '01-701-1148', '01-710-1249', '01-701-1415', '01-701-1146', '01-703-1403', '01-709-1312', '01-716-1167', '01-708-1253', '01-713-1256', '01-713-1073', '01-716-1177', '01-711-1251', '01-709-1088', '01-709-1099', '01-703-1396', '01-701-1317', '01-709-1168', '01-710-1137', '01-708-1184', '01-715-1407', '01-718-1371', '01-710-1021', '01-716-1108', '01-709-1259', '01-701-1392', '01-701-1047', '01-703-1379', '01-704-1260', '01-709-1306', '01-701-1411', '01-707-1434', '01-710-1083', '01-708-1297', '01-715-1134', '01-710-1060', '01-718-1172', '01-703-1197', '01-705-1011', '01-704-1025', '01-708-1087', '01-701-1015', '01-710-1027', '01-717-1

# データの書き換え

In [8]:
data = [
["DM", "01-703-1096",   0, "AGE", 49],
["LB", "01-703-1042",   3, "LBORRES", "135"],
["LB", "01-703-1042",   4, "LBORRES", "145"],
["LB", "01-703-1086",  37, "LBORRES", "1"],
["LB", "01-703-1086",  72, "LBORRES", "1.2"],
["LB", "01-703-1086", 102, "LBORRES", "1.1"],
["LB", "01-703-1086", 132, "LBORRES", "1"],
["LB", "01-703-1086", 162, "LBORRES", "1.3"],
["LB", "01-703-1086", 197, "LBORRES", "0.9"],
["LB", "01-703-1086", 232, "LBORRES", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESC", "135"],
["LB", "01-703-1042",   4, "LBSTRESC", "145"],
["LB", "01-703-1086",  37, "LBSTRESC", "1"],
["LB", "01-703-1086",  72, "LBSTRESC", "1.2"],
["LB", "01-703-1086", 102, "LBSTRESC", "1.1"],
["LB", "01-703-1086", 132, "LBSTRESC", "1"],
["LB", "01-703-1086", 162, "LBSTRESC", "1.3"],
["LB", "01-703-1086", 197, "LBSTRESC", "0.9"],
["LB", "01-703-1086", 232, "LBSTRESC", "0.8"],
["LB", "01-703-1042",   3, "LBSTRESN", 135],
["LB", "01-703-1042",   4, "LBSTRESN", 145],
["LB", "01-703-1086",  37, "LBSTRESN", 1],
["LB", "01-703-1086",  72, "LBSTRESN", 1.2],
["LB", "01-703-1086", 102, "LBSTRESN", 1.1],
["LB", "01-703-1086", 132, "LBSTRESN", 1],
["LB", "01-703-1086", 162, "LBSTRESN", 1.3],
["LB", "01-703-1086", 197, "LBSTRESN", 0.9],
["LB", "01-703-1086", 232, "LBSTRESN", 0.8],
["LB", "01-703-1042",   3, "LBNRIND", "HIGH"],
["LB", "01-703-1042",   4, "LBNRIND", "HIGH"],
["LB", "01-703-1086",  37, "LBNRIND", "LOW"],
["LB", "01-703-1086",  72, "LBNRIND", "LOW"],
["LB", "01-703-1086", 102, "LBNRIND", "LOW"],
["LB", "01-703-1086", 132, "LBNRIND", "LOW"],
["LB", "01-703-1086", 162, "LBNRIND", "LOW"],
["LB", "01-703-1086", 197, "LBNRIND", "LOW"],
["LB", "01-703-1086", 232, "LBNRIND", "LOW"],
["MH", "01-701-1097",   1, "MHTERM", "Loss of consciousness (Passed out)"],
["MH", "01-701-1097",   1, "MHSTDTC", "2023-01-01"],
["MH", "01-701-1111",   1, "MHTERM", "HEARING LOSS"],
["MH", "01-701-1180",   1, "MHTERM", "DEPRESSION (ANXIETY)"],
["MH", "01-702-1082",   1, "MHTERM", "Premenstrual pain"],
["MH", "01-703-1076",   1, "MHTERM", "Atrioventricular block (scheduled cardiac pacemaker insertion)"],
["MH", "01-703-1279",   1, "MHTERM", "schizophreniform disorders"],
["MH", "01-703-1299",   1, "MHTERM", "Cyclothymic disorder"],
["VS", "01-701-1047",  17, "VSORRES", "121"],
["VS", "01-701-1047",  18, "VSORRES", "124"],
["VS", "01-701-1047",  66, "VSORRES", "185"],
["VS", "01-701-1047",  67, "VSORRES", "183"],
["VS", "01-701-1383",  37, "VSORRES", "98"],
["VS", "01-701-1383", 122, "VSORRES", "160"],
["VS", "01-701-1387",   1, "VSORRES", "146"],
["VS", "01-701-1387",  32, "VSORRES", "72"],
["VS", "01-701-1047",  17, "VSSTRESC", "121"],
["VS", "01-701-1047",  18, "VSSTRESC", "124"],
["VS", "01-701-1047",  66, "VSSTRESC", "185"],
["VS", "01-701-1047",  67, "VSSTRESC", "183"],
["VS", "01-701-1383",  37, "VSSTRESC", "98"],
["VS", "01-701-1383", 122, "VSSTRESC", "160"],
["VS", "01-701-1387",   1, "VSSTRESC", "146"],
["VS", "01-701-1387",  32, "VSSTRESC", "72"],
["VS", "01-701-1047",  17, "VSSTRESN", 121],
["VS", "01-701-1047",  18, "VSSTRESN", 124],
["VS", "01-701-1047",  66, "VSSTRESN", 185],
["VS", "01-701-1047",  67, "VSSTRESN", 183],
["VS", "01-701-1383",  37, "VSSTRESN", 98],
["VS", "01-701-1383", 122, "VSSTRESN", 160],
["VS", "01-701-1387",   1, "VSSTRESN", 146],
["VS", "01-701-1387",  32, "VSSTRESN", 72],
["EX", "01-701-1148",   2, "EXDOSE", 82],
["EX", "01-701-1148",   3, "EXDOSE", 216],
["EX", "01-703-1258",   2, "EXDOSE", 27],
["CM", "01-701-1146",  29, "CMTRT", "PAROXETINE"],
["QS", "01-701-1023",1010, "QSORRES", "PRESENT"],
["QS", "01-701-1023",1012, "QSORRES", "PRESENT"],
["QS", "01-701-1111",5004, "QSORRES", "4"],
["QS", "01-701-1111",5019, "QSORRES", "4"],
["QS", "01-701-1111",5012, "QSORRES", "4"],
["QS", "01-701-1111",5027, "QSORRES", "4"],
["QS", "01-701-1118",6002, "QSORRES", "MARKED IMPROVEMENT"],
["QS", "01-701-1118",6003, "QSORRES", "MARKED WORSENING"],
["QS", "01-701-1181",4018, "QSORRES", "Y"],
["QS", "01-701-1181",4058, "QSORRES", "Y"],
["QS", "01-701-1181",4019, "QSORRES", "Y"],
["QS", "01-701-1181",4059, "QSORRES", "Y"],
["QS", "01-701-1181",4020, "QSORRES", "Y"],
["QS", "01-701-1023",1010, "QSSTRESC", "2"],
["QS", "01-701-1023",1012, "QSSTRESC", "2"],
["QS", "01-701-1111",5004, "QSSTRESC", "4"],
["QS", "01-701-1111",5019, "QSSTRESC", "4"],
["QS", "01-701-1111",5012, "QSSTRESC", "4"],
["QS", "01-701-1111",5027, "QSSTRESC", "4"],
["QS", "01-701-1118",6002, "QSSTRESC", "1"],
["QS", "01-701-1118",6003, "QSSTRESC", "7"],
["QS", "01-701-1181",4018, "QSSTRESC", "1"],
["QS", "01-701-1181",4058, "QSSTRESC", "1"],
["QS", "01-701-1181",4019, "QSSTRESC", "1"],
["QS", "01-701-1181",4059, "QSSTRESC", "1"],
["QS", "01-701-1181",4020, "QSSTRESC", "1"],
["QS", "01-701-1023",1010, "QSSTRESN", 2],
["QS", "01-701-1023",1012, "QSSTRESN", 2],
["QS", "01-701-1111",5004, "QSSTRESN", 4],
["QS", "01-701-1111",5019, "QSSTRESN", 4],
["QS", "01-701-1111",5012, "QSSTRESN", 4],
["QS", "01-701-1111",5027, "QSSTRESN", 4],
["QS", "01-701-1118",6002, "QSSTRESN", 1],
["QS", "01-701-1118",6003, "QSSTRESN", 7],
["QS", "01-701-1181",4018, "QSSTRESN", 1],
["QS", "01-701-1181",4058, "QSSTRESN", 1],
["QS", "01-701-1181",4019, "QSSTRESN", 1],
["QS", "01-701-1181",4059, "QSSTRESN", 1],
["QS", "01-701-1181",4020, "QSSTRESN", 1],
["QS", "01-701-1118",6001, "QSDTC", "2014-07-08"],
["QS", "01-701-1118",6001, "QSDY", 119],
["AE", "01-701-1015",   3, "AESER", "Y"],
["AE", "01-701-1015",   3, "AESHOSP", "Y"],
["AE", "01-701-1015",   3, "AESTDTC", "2014-01-11"],
["AE", "01-701-1015",   3, "AEENDTC", "2014-01-09"],
["AE", "01-701-1015",   3, "AESTDY", 10],
["AE", "01-701-1015",   3, "AEENDY", 8],
["AE", "01-701-1028",   1, "AETERM", "PARKINSON'S DISEASE"],
["AE", "01-701-1028",   1, "AESTDTC", "2013-07-01"],
["AE", "01-701-1028",   1, "AESTDY", -17],
["AE", "01-701-1034",   2, "AETERM", "MALIGNANT HYPERTENSION"],
["AE", "01-701-1047",   4, "AETERM", "HYPERTENSION"],
["AE", "01-701-1363",   1, "AESTDTC", "2013-06-15"],
["AE", "01-701-1363",   1, "AEENDTC", "2013-06-14"],
["AE", "01-701-1363",   1, "AESTDY", 17],
["AE", "01-701-1363",   1, "AEENDY", 16],
["AE", "01-701-1047",   3, "AEENDTC", "2013-03-05"],
["AE", "01-701-1047",   3, "AEENDY", 22],
["AE", "01-701-1383",  12, "AETERM", "BLOOD PRESSURE INCREASED"],
["AE", "01-701-1153",   2, "AEACN", "DRUG WITHDRAWN"],
["AE", "01-701-1180",   6, "AETERM", "SUDDEN DEATH"],
["AE", "01-703-1258",   2, "AESEV", "SEVERE"],
["AE", "01-703-1258",   2, "AESTDTC", "2012-08-01"],
["AE", "01-703-1258",   2, "AEENDTC", "2012-10-01"],
["AE", "01-703-1258",   2, "AESTDY", 13],
["AE", "01-703-1258",   2, "AEENDY", 74],
["AE", "01-703-1258",   5, "AESEV", "MODERATE"],
["AE", "01-703-1258",   5, "AESER", "Y"],
["AE", "01-703-1258",   5, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-703-1258",   5, "AESLIFE", "Y"],
["AE", "01-703-1258",   5, "AESTDTC", "2012-10-02"],
["AE", "01-703-1258",   5, "AEENDTC", "2012-12-31"],
["AE", "01-703-1258",   2, "AESTDY", 75],
["AE", "01-703-1258",   2, "AEENDY", 165],
["AE", "01-703-1335",   1, "AETERM", "MULTIPLE SCLEROSIS RELAPSE"],
["AE", "01-703-1335",   1, "AESTDTC", "2014-04-01"],
["AE", "01-703-1335",   1, "AEENDTC", "2014-05-01"],
["AE", "01-703-1335",   1, "AESTDY", 15],
["AE", "01-703-1335",   1, "AEENDY", 46],
["AE", "01-703-1403",   2, "AETERM", "MYASTHENIA GRAVIS AGGRAVATED"],
["AE", "01-704-1008",   1, "AETERM", "TREMOR IN HANDS, LEGS"],
["AE", "01-704-1008",   1, "AEREL", "NONE"],
["AE", "01-704-1008",   1, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   1, "AESTDY", -225],
["AE", "01-704-1008",   3, "AETERM", "MUSCLE STIFFNESS"],
["AE", "01-704-1008",   3, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   3, "AESTDY", -225],
["AE", "01-704-1008",   2, "AETERM", "SLOWNESS of MOVEMENT"],
["AE", "01-704-1008",   2, "AESTDTC", "2012-06-01"],
["AE", "01-704-1008",   2, "AESTDY", -225],
["AE", "01-704-1009",   6, "AETERM", "CHRONIC KIDNEY DISEASE"],
["AE", "01-704-1009",   6, "AESER", "Y"],
["AE", "01-704-1009",   6, "AESLIFE", "Y"],
["AE", "01-704-1010",   1, "AETERM", "DIABETES MELLITUS"],
["AE", "01-704-1010",   1, "AESER", "Y"],
["AE", "01-704-1010",   1, "AESLIFE", "Y"],
["AE", "01-704-1017",   4, "AETERM", "LATE EFFECTS OF CEREBRAL INFRACTION"],
["AE", "01-704-1017",   4, "AESEV", "SEVERE",],
["AE", "01-704-1017",   4, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   4, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   4, "AESTDY", 14],
["AE", "01-704-1017",   4, "AEENDY", 44],
["AE", "01-704-1017",   3, "AETERM", "BRAIN DEATH"],
["AE", "01-704-1017",   3, "AESEV", "SEVERE",],
["AE", "01-704-1017",   3, "AESTDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AEENDTC", "2013-11-18"],
["AE", "01-704-1017",   3, "AESTDY", 44],
["AE", "01-704-1017",   3, "AEENDY", 44],
["AE", "01-704-1017",   1, "AEOUT", "RECOVERED/RESOLVED"],
["AE", "01-704-1017",   1, "AESTDTC", "2013-10-19"],
["AE", "01-704-1017",   1, "AEENDTC", "2013-11-19"],
["AE", "01-704-1017",   1, "AESTDY", 14],
["AE", "01-704-1017",   1, "AEENDY", 45],
["AE", "01-704-1017",   1, "AEACN", "DRUG WITHDRAWN"]
]

target = dataset_list

for l in data:
  #print(l)
  target = data_update(target, l[0], l[1], l[2], l[3], l[4])

USUBJID '01-703-1096' の 'AGE' を '49' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBORRES' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBORRES' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBORRES' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBORRES' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBORRES' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '162' の 'LBORRES' を '1.3' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '197' の 'LBORRES' を '0.9' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '232' の 'LBORRES' を '0.8' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '3' の 'LBSTRESC' を '135' に更新しました。
USUBJID '01-703-1042'、'LBSEQ' '4' の 'LBSTRESC' を '145' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '37' の 'LBSTRESC' を '1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '72' の 'LBSTRESC' を '1.2' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '102' の 'LBSTRESC' を '1.1' に更新しました。
USUBJID '01-703-1086'、'LBSEQ' '132' の 'LBSTRESC' を '1' に更

In [9]:
# 更新データ確認
compare_data(dataset_list, target)

--- ItemGroupOID: AE ---
USUBJID = 01-701-1015, AESEQ = 3:
  Updated:
    AEENDTC: '2014-01-11' -> '2014-01-09'
    AEENDY: 10 -> 8
    AESER: 'N' -> 'Y'
    AESHOSP: 'N' -> 'Y'
    AESTDTC: '2014-01-09' -> '2014-01-11'
    AESTDY: 8 -> 10

USUBJID = 01-701-1028, AESEQ = 1:
  Updated:
    AESTDTC: '2013-07-21' -> '2013-07-01'
    AESTDY: 3 -> -17
    AETERM: 'APPLICATION SITE ERYTHEMA' -> "PARKINSON'S DISEASE"

USUBJID = 01-701-1034, AESEQ = 2:
  Updated:
    AETERM: 'FATIGUE' -> 'MALIGNANT HYPERTENSION'

USUBJID = 01-701-1047, AESEQ = 3:
  Updated:
    AEENDTC: '' -> '2013-03-05'
    AEENDY: None -> 22

USUBJID = 01-701-1047, AESEQ = 4:
  Updated:
    AETERM: 'BUNDLE BRANCH BLOCK LEFT' -> 'HYPERTENSION'

USUBJID = 01-701-1153, AESEQ = 2:
  Updated:
    AEACN: '' -> 'DRUG WITHDRAWN'

USUBJID = 01-701-1180, AESEQ = 6:
  Updated:
    AETERM: 'MICTURITION URGENCY' -> 'SUDDEN DEATH'

USUBJID = 01-701-1363, AESEQ = 1:
  Updated:
    AEENDTC: '2013-06-15' -> '2013-06-14'
    AEENDY: 17 -> 16

# LLMへの送信

In [10]:
!pip install sseclient-py

In [11]:
import requests
import sseclient

In [12]:
def run_dify_workflow(api_key, workflow_inputs, user_id, response_mode='streaming'):
    url = 'https://api.dify.ai/v1/workflows/run'
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json'
    }

    payload = {
        'inputs': workflow_inputs,
        'response_mode': response_mode,
        'user': user_id
    }

    if response_mode == 'blocking':
        response = requests.post(url, headers=headers, json=payload)
        return response.json()
    elif response_mode == 'streaming':
        response = requests.post(url, headers=headers, json=payload, stream=True)
        client = sseclient.SSEClient(response)
        return client.events()


def print_human_readable(data):
    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, str):
                print(f"{key}: {value}")
            elif isinstance(value, dict):
                print(f"{key}:")
                print_human_readable(value)
            else:
                print(f"{key}: {value}")
    elif isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, ensure_ascii=False, indent=2))

In [13]:
updated_subjects = []
for l in data:
  updated_subjects.append(l[1])

updated_subjects = list(set(updated_subjects))
updated_subjects

['01-704-1017',
 '01-703-1076',
 '01-701-1097',
 '01-704-1010',
 '01-703-1096',
 '01-704-1009',
 '01-701-1023',
 '01-701-1034',
 '01-703-1279',
 '01-703-1042',
 '01-701-1383',
 '01-701-1180',
 '01-701-1148',
 '01-701-1146',
 '01-701-1181',
 '01-703-1403',
 '01-701-1111',
 '01-701-1363',
 '01-704-1008',
 '01-701-1047',
 '01-703-1335',
 '01-701-1387',
 '01-701-1015',
 '01-701-1118',
 '01-703-1086',
 '01-701-1028',
 '01-703-1258',
 '01-703-1299',
 '01-701-1153',
 '01-702-1082']

In [14]:
with open('define_xml/define.xml', 'r') as f:
  define_xml = f.read()


SysPrompt = '''
あなたは、臨床試験データのレビューを支援するAIアシスタントです。以下の前提知識を理解した上で、ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援してください。

**前提知識:**

*   **臨床試験においては参加者の安全性が何よりも重要視されます。有害事象の評価は特に重要なデータとなります。**
*   **SDTM (Study Data Tabulation Model) とは:** 臨床試験データを標準化するためのデータモデルであり、CDISC (Clinical Data Interchange Standards Consortium) によって策定されていること。
*   **Define.xml とは:** SDTMデータの構造や変数定義を記述したメタデータファイルであること。**Define.xmlはデータ構造を記述する「説明書」であり、データ自体を制約するものではありません。データがDefine.xmlの定義に完全に一致していなくても、それは必ずしもエラーではありません。**
*   **SDTMのドメイン:** SDTMデータは、患者背景(DM)、有害事象(AE)、バイタルサイン(VS)、検査値(LB)など、複数のドメイン（データセット）に分かれていること。各ドメインには、特定の変数（列）が含まれていること。
*   **報告されているJSONデータにはデータ入力時の間違いが含まれる可能性があります。**

**出力形式:**

*   すべての出力はMarkdown形式で作成してください。表形式は使用しないでください。
'''

UserInput_initiate = '''
臨床試験データ、Define.xml、およびプロトコルを基に、以下に示すタスクを実行してください。必要なデータは、このプロンプトの最後にまとめて記載しています。
'''



UserInput_Task1 = '''
**役割:** あなたは臨床試験の専門医です。以下のタスクを実行してください。

1.  **JSONデータのレビュー:**
    *   提供される情報から、対象となる臨床試験のフェーズ、疾患領域、有効性および安全性の評価項目を特定してください。
    *   後述するJSONデータ（SDTM形式）、Define.xmlファイル（参考情報）、およびプロトコルを確認してください。
    *   **Define.xmlは参考情報として活用し、データそのものの内容、医学的妥当性、プロトコルとの整合性を重視してレビューしてください。**
    *   **上記を踏まえてJSONデータをレビューし、症例サマリーを作成してください。**

2.  **クエリの作成:** (必要な場合)
    *   JSONデータをレビューする中で、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは報告されたデータに基づいて作成してください。想像やハルシネーションに基づいたクエリは作成してはいけません。
    *   クエリは臨床試験の評価項目に対する影響度を考慮して作成する必要があります。重要なクエリを優先的に作成してください。

**JSONデータのレビュー観点:**

*   **安全性:**
    *   有害事象(AEドメイン)の報告内容は、医学的に妥当であるか？
*   **医学的妥当性:**
    *   検査値(LBドメイン)の変動、バイタルサイン(VSドメイン)の変動、併用薬(CMドメイン)との相互作用など、時間経過とともに医学的に問題となる点は見られるか？
*   **有効性:**
    *   特定された主要評価項目および副次評価項目について、その時間的変化は期待される効果と一致しているか？
*   **その他:**
    *   患者背景(DMドメイン)、既往歴(MHドメイン)、治療歴(EXドメイン, CMドメイン)などを総合的に考慮し、時間経過を加味して安全性に懸念を生じる事項があれば記載してください。
*   **プロトコル逸脱 (疑い):**
    *   選択/除外基準、投与量、併用禁止薬、評価スケジュール、有害事象報告などについて、プロトコルからの逸脱の疑いがないか確認してください。（関連ドメイン: DM, MH, EX, CM, LB, VS, AEなど）

**試験情報まとめのテンプレート:**

*   **試験フェーズ:**
*   **疾患領域:**
*   **有効性評価項目:**
    *   **主要評価項目:**
    *   **副次評価項目:**
*   **安全性評価項目:**

**症例サマリーのテンプレート:**

*   **患者ID:**
    *   YYYY年MM月DD日 (Day XX):

**クエリのテンプレート:**（必要な場合）

＜以下のMarkdownの箇条書きを所見ごとに繰り返し使用してください＞

*   **患者ID:**
    *   **臨床試験結果への影響度合い:**（Critical/Major/Minor/None）
    *   **関連ドメイン/変数:**
    *   **医療機関への問い合わせ文面:**
    *   **判断理由:**

**備考:**

*   各イベントは、Define.xmlに定義された日付変数などを参考に、正確な日時を特定してください。
'''



UserInput_Task2 = '''
**役割:** あなたはクリニカルデータマネージャーです。以下のタスクを実行してください。

**タスク:**

1.  **JSONデータのレビュー:**
    *   後述するJSONデータ（SDTM形式）、Define.xmlファイル（参考情報）、およびプロトコルを確認してください。
    *   **特に、異なるSDTMドメイン間のデータの整合性に焦点を当ててレビューしてください。**
    *   **Define.xmlは参考情報として活用し、データそのものの整合性、プロトコルとの整合性を重視してください。Define.xmlとデータの間に不整合がある場合は、「Define.xmlの修正候補」として報告してください。**

2.  **クエリの作成:** (必要な場合)
    *   JSONデータをレビューする中で、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは報告されたデータに基づいて作成してください。想像やハルシネーションに基づいたクエリは作成してはいけません。
    *   クエリは臨床試験の評価項目に対する影響度を考慮して作成する必要があります。重要なクエリを優先的に作成してください。

**JSONデータのレビュー観点:**

1.  **クロスドメイン整合性:**
    *   異なるSDTMドメイン間で、データに矛盾がないか？
    *   ドメイン間の関連性が正しく表現されているか？
    *   **具体的な確認例 (これらに限定されない):**
        *   DM.SEXとAEにおける妊娠関連の有害事象
        *   AEの有害事象発現日や治験薬との関連性と、EXの治験薬の投与期間
        *   LBの検査値異常とAEの関連有害事象
        *   VSのバイタルサイン異常とAEの関連有害事象
        *   CM.CMTRTとAE/MHで報告されている疾患・既往歴との矛盾

2.  **単一ドメイン内の整合性:**
    *   Define.xmlの定義に照らして、矛盾なく解釈できるデータになっているか？
    *   プロトコルに照らして、データの関連性が正しく表現されているか？

3.  **異常値:**
    *   Define.xmlで定義された範囲外、または医学的にありえない値がないか？

4.  **欠損値:**
    *   欠損値の有無と理由。多い場合は原因推測。

5.  **プロトコル逸脱 (データ品質の観点から):**
    *   データ入力/収集で、プロトコルからの逸脱（例：必須項目の未入力、不適切な時期のデータ収集）がないか？

**データ品質報告書のテンプレート:**

*   **全体的なデータ品質の評価:**
    *   総合評価:
    *   データクリーニング/再調査が必要な項目:

*   **クロスドメイン整合性に関する問題点:**（問題がある場合）
    *   関連するドメイン:
    *   問題点の詳細:
        *   変数名:
        *   具体的な矛盾:
        *   該当レコード数/割合:
    *   問題点の原因（推測）:
    *   対応策（提案）:

*   **単一ドメイン内の整合性に関する問題点:**（問題がある場合）
    *   ドメイン名:
    *   問題点の種類:
    *   問題点の詳細:
    *   問題点の原因（推測）:
    *   対応策（提案）:

*   **欠損値、異常値、プロトコル逸脱(データ品質関連)に関する問題点:** （問題がある場合）
    *   ドメイン名:
    *   問題点の種類:
    *   問題点の詳細:
    *   問題点の原因（推測）:
    *   対応策（提案）:

*   **その他:**（問題がある場合）
    *   **Define.xmlとデータの不整合（Define.xmlの修正候補）:** Define.xmlとデータの間に不整合がある場合は、以下のように記載してください。
        *   **不整合箇所:** (例: ドメイン名、変数名)
        *   **データの値:**
        *   **Define.xmlの定義:**
        *   **修正提案:** (Define.xmlをどのように修正すべきか)
    * プロトコルに照らし合わせて、データ収集手順に問題がある場合は、その旨と改善提案を記載する。

**クエリのテンプレート:**（必要な場合）

＜以下のMarkdownの箇条書きを所見ごとに繰り返し使用してください＞

*   **患者ID:**
    *   **臨床試験結果への影響度合い:**（Critical/Major/Minor/None）
    *   **関連ドメイン/変数:**
    *   **医療機関への問い合わせ文面:**
    *   **判断理由:**

'''


UserInput_Task3 = '''
**役割:** あなたは臨床試験の専門医およびデータマネージャーの視点を持つ、プロトコル遵守状況の確認者です。

**タスク:**

1.  **プロトコル逸脱の検出:**
    *   後述するJSONデータ（SDTM形式）、Define.xmlファイル（参考情報）、およびプロトコルを参照し、プロトコルからの逸脱を検出してください。
    *   **Define.xmlは参考情報として活用し、データとプロトコルの内容を比較して逸脱を判断してください。**

2.  **クエリの作成:** (必要な場合)
    *   **プロトコル逸脱を判定するために**医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは報告されたデータに基づいて作成してください。想像やハルシネーションに基づいたクエリは作成してはいけません。
    *   クエリはプロトコル逸脱が与える評価項目への影響度を考慮して作成する必要があります。重要なクエリを優先的に作成してください。

**検出対象とすべき主要なプロトコル逸脱の例 (これらに限定されない):**

*   **選択/除外基準違反:** (関連SDTMドメイン: DM, MH など)
*   **投与量違反:** (関連SDTMドメイン: EX)
*   **併用禁止薬の使用:** (関連SDTMドメイン: CM)
*   **評価スケジュール違反:** (関連SDTMドメイン: LB, VS, その他)
*   **有害事象報告違反**: (関連SDTMドメイン: AE)

**プロトコル逸脱報告書のテンプレート:**

＜以下のMarkdownの箇条書きを所見ごとに繰り返し使用してください＞

*   **患者ID:**
    *   **逸脱の分類:**（Critical/Major/Minor）
    *   **関連ドメイン/変数:**
    *   **逸脱内容:**
    *   **プロトコル該当箇所:**
    *   **判断理由:**

**クエリのテンプレート:**（必要な場合）

＜以下のMarkdownの箇条書きを所見ごとに繰り返し使用してください＞

*   **患者ID:**
    *   **臨床試験結果への影響度合い:**（Critical/Major/Minor/None）
    *   **関連ドメイン/変数:**
    *   **医療機関への問い合わせ文面:**
    *   **判断理由:**
'''


UserInput_end1 = '''\n---\n\n**データ:**\n\n*   臨床試験データ（JSON形式、SDTM準拠）:\n\n```json\n'''
UserInput_end2 = '''\n```\n\n*   データ定義ファイル（Define.xml）:\n\n```xml\n''' + define_xml + '''```\n'''

In [16]:
from IPython.display import display, Markdown

#ModelName = 'gemini-2.0-flash'
#ModelName = 'gemini-2.0-flash-exp'
#ModelName = 'gemini-2.0-pro-exp-02-05'
ModelName = 'gemini-2.0-flash-thinking-exp-01-21'
#ModelName = 'gemini-2.0-flash-thinking-exp'

datasetjson = filter_data(target, updated_subjects[8])
#datasetjson = filter_data(target, list(usubjids)[100])


# 使用例
api_key = 'app-mngwx1rfnUH26Pt1a8UVpMLU'
#workflow_inputs = {'ModelName': ModelName, 'SysPrompt': SysPrompt, 'UserInput': UserInput_initiate + UserInput_Task3 + UserInput_end1 + json.dumps(datasetjson) + UserInput_end2, 'AttachedFile': {"type": "document", "transfer_method": "remote_url", "url": "https://wiki.ihe.net/images/4/47/Lzzt_protocol_redacted.pdf"}}  # ワークフローの入力パラメータ
workflow_inputs = {'ModelName': ModelName, 'SysPrompt': SysPrompt, 'UserInput': UserInput_initiate + UserInput_Task3 + UserInput_end1 + json.dumps(datasetjson) + UserInput_end2, 'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}}  # ワークフローの入力パラメータ
#workflow_inputs = {'ModelName': 'gemini-2.0-flash', 'UserInput': 'こんにちは'}  # ワークフローの入力パラメータ
user_id = 'user123'

# ブロッキングモードの例
#result = run_dify_workflow(api_key, workflow_inputs, user_id, 'blocking')
#print_human_readable(result)


#display(Markdown(result['data']['outputs']['text']))

# ストリーミングモードの例
for event in run_dify_workflow(api_key, workflow_inputs, user_id, response_mode='streaming'):
    print_human_readable(json.loads(event.data))

display(Markdown(json.loads(event.data)['data']['outputs']['text']))

ストリーミング出力は最後の 5000 行に切り捨てられました。
    </Decode>
  </CodeListItem>
  <CodeListItem CodedValue="9" def:Rank="9">
    <Decode>
      <TranslatedText xml:lang="en">9</TranslatedText>
    </Decode>
  </CodeListItem>
</CodeList>
<CodeList OID="SCUNIT" Name="SCUNIT" DataType="text">
  <CodeListItem CodedValue="YEARS" def:Rank="1">
    <Decode>
      <TranslatedText xml:lang="en">YEARS</TranslatedText>
    </Decode>
  </CodeListItem>
</CodeList>
<CodeList OID="SEV" Name="SEV" DataType="text">
  <CodeListItem CodedValue="MILD" def:Rank="1">
    <Decode>
      <TranslatedText xml:lang="en">MILD</TranslatedText>
    </Decode>
  </CodeListItem>
  <CodeListItem CodedValue="MODERATE" def:Rank="2">
    <Decode>
      <TranslatedText xml:lang="en">MODERATE</TranslatedText>
    </Decode>
  </CodeListItem>
  <CodeListItem CodedValue="SEVERE" def:Rank="3">
    <Decode>
      <TranslatedText xml:lang="en">SEVERE</TranslatedText>
    </Decode>
  </CodeListItem>
</CodeList>
<CodeList OID="SEVSC" Name="SEVSC" 

*   **患者ID:** 01-703-1279
    *   **逸脱の分類:** Critical
    *   **関連ドメイン/変数:** QS/MMITM05, TI/INCL03, QS/MMTOT
    *   **逸脱内容:** MMSEスコアが0点であり、組み入れ基準のMMSEスコア10〜23点を満たしていません。
    *   **プロトコル該当箇所:** 3.4.2.1 Inclusion Criteria [3]
    *   **判断理由:**  QSドメインのMMITM05変数の値が0であり、プロトコルで規定された組み入れ基準（MMSEスコア10〜23点）を満たしていないため、重大なプロトコル逸脱と判断しました。MMSEは認知機能を評価する主要な評価項目であり、組み入れ基準を満たさない患者のデータは試験結果の解釈に影響を与える可能性があります。

*   **患者ID:** 01-703-1279
    *   **逸脱の分類:** Major
    *   **関連ドメイン/変数:** MH/MHTERM, TI/EXCL14
    *   **逸脱内容:** 過去5年以内の精神疾患の既往歴の可能性（統合失調症様障害）
    *   **プロトコル該当箇所:** 3.4.2.2. Exclusion Criteria [14] A history within the last 5 years of the following: a) Schizophrenia b) Bipolar Disease c) Ethanol or psychoactive drug abuse or dependence.
    *   **判断理由:** 医療既往歴(MH)ドメインに「schizophreniform disorders (統合失調症様障害)」の記載があり、除外基準EXCL14「A history of mental illness within the last 5 years. (過去5年以内の精神疾患歴)」に抵触する可能性があります。

*   **患者ID:** 01-703-1279
    *   **逸脱の分類:** Major
    *   **関連ドメイン/変数:** LB/LBTESTCD, LB/LBORRES, TI/EXCL27
    *   **逸脱内容:** 臨床検査値異常（Hematocrit, Hemoglobin 高値）による除外基準違反の可能性
    *   **プロトコル該当箇所:** 3.4.2.2. Exclusion Criteria [27b] Laboratory test values exceeding the Lilly Reference Range III for the patient’s age in any of the following analytes: creatinine, total bilirubin, SGOT, SGPT, etc.  ↑↓ hemoglobin, ↑↓ white blood cell count, ↑↓ platelet count, ↑↓ serum sodium, potassium, or calcium.
    *   **判断理由:**  LBドメインのデータにおいて、HCT (Hematocrit) および HGB (Hemoglobin) が基準範囲上限を超過しており、プロトコル除外基準EXCL27bに抵触する可能性があります。

*   **患者ID:** 01-703-1279
    *   **逸脱の分類:** Major
    *   **関連ドメイン/変数:** EX/EXENDTC, DS/DSSTDTC, SE/SEENDTC, TE/TEDUR
    *   **逸脱内容:** 治験薬投与期間がプロトコルで規定された期間より短い（26週間より大幅に短い）
    *   **プロトコル該当箇所:** 3.1. Summary of Study Design, 3.6.2. TTS Administration Procedures, TE/TEDUR
    *   **判断理由:** プロトコルおよびTEドメインでは26週間の投与期間が計画されているにもかかわらず、EXドメイン、DSドメイン、SEドメインのデータから、実際の投与期間が大幅に短いことが示唆されます。DSドメインの治験中止理由から、被験者による治験からのWithdrawalが原因と考えられます。

*   **患者ID:** 01-703-1279
    *   **逸脱の分類:** Minor
    *   **関連ドメイン/変数:** VS/VSTESTCD=DIABP, VSTESTCD=PULSE, VSPOS=STANDING, VISIT=WEEK 2
    *   **逸脱内容:** 評価項目（バイタルサイン）の未実施（WEEK 2の立位血圧(3分後)および脈拍(3分後)）
    *   **プロトコル該当箇所:** 3.9.3.4.1 Vital Sign Determination, Protocol Attachment LZZT.1. Schedule of Events for Protocol H2Q-MC-LZZT(c)
    *   **判断理由:** VSドメインにおいて、WEEK 2 (VISITNUM=4) の立位3分後の血圧と脈拍のデータが欠損 (VSSTAT=NOT DONE) しており、プロトコルで規定された評価項目の一部が実施されていない可能性があります。

## クエリ

*   **患者ID:** 01-703-1279
    *   **臨床試験結果への影響度合い:** Critical
    *   **関連ドメイン/変数:** QS/MMITM04, QS/MMITM05, TI/INCL03, QS/MMTOT
    *   **医療機関への問い合わせ文面:**
        *   患者ID 01-703-1279 のスクリーニング1 (VISITNUM=1) におけるミニメンタルステート検査 (MMSE) の合計スコアと、各項目のスコア（特にMMITM01-MMITM06）を教えてください。
        *   もしMMSE合計スコアが10点未満であった場合、inclusion criteria INCL03 (MMSE score of 10 to 23) を満たしていなかった理由と、組み入れ判断の根拠について説明してください。
    *   **判断理由:** 患者ID 01-703-1279 のMMSEスコアが選択基準を満たしているか否かを確認する必要があるため。MMSEスコアは主要な選択基準の一つであり、逸脱していた場合、試験結果の解釈に影響を与える可能性があります。

*   **患者ID:** 01-703-1279
    *   **臨床試験結果への影響度合い:** Major
    *   **関連ドメイン/変数:** MH/MHTERM
    *   **医療機関への問い合わせ文面:**
        *   患者ID 01-703-1279 の医療既往歴について確認させてください。Medical Historyドメインに「schizophreniform disorders」とありますが、これはSchizophrenia (統合失調症) または Schizophreniform disorder (統合失調症様障害) の既往歴を意味するものでしょうか。もしそうであれば、プロトコルの除外基準 [14] に抵触する可能性があります。
        *   もし「schizophreniform disorders」が誤記であり、関節炎(ARTHRITIS) を意味するのであれば、その旨をご回答ください。また、関節炎と記録されている場合、MHTERMの "schizophreniform disorders" はタイプミスであると考えられますが、データ修正は可能でしょうか。
    *   **判断理由:** 統合失調症または統合失調症様障害の既往歴は、プロトコルの主要な除外基準であり、試験対象患者の適格性に大きく影響を与えるため。

*   **患者ID:** 01-703-1279
    *   **臨床試験結果への影響度合い:** Major
    *   **関連ドメイン/変数:** LB/LBTESTCD, LB/LBORRES
    *   **医療機関への問い合わせ文面:**
        *   患者ID 01-703-1279 の臨床検査値 (Laboratory Tests Resultsドメイン) について確認させてください。
        *   Hematocrit (HCT) および Hemoglobin (HGB) の値が基準範囲上限を超過していますが、除外基準 [27b] 適用において、Lilly Reference Range III における基準範囲からの逸脱と判断されたかどうかご教示ください。
        *   もし逸脱と判断された場合、除外基準抵触時の対応手順について、治験事務局の見解をご教示ください。
    *   **判断理由:** 臨床検査値の異常は、プロトコルの主要な除外基準に該当する可能性があり、試験対象患者の適格性や安全性評価に影響を与える可能性があるため。

*   **患者ID:** 01-703-1279
    *   **臨床試験結果への影響度合い:** Minor
    *   **関連ドメイン/変数:** VS/VISIT, VS/VSTESTCD
    *   **医療機関への問い合わせ文面:**
        *   患者ID 01-703-1279 のバイタルサイン測定 (Vital Signsドメイン) について確認させてください。
        *   Week 2 (VISITNUM=4) の訪問において、立位でのDiastolic Blood Pressure (DIABP) および Pulse Rate (PULSE) の測定 (VSTPT=AFTER STANDING FOR 3 MINUTES) が "NOT DONE" と記録されていますが、測定が実施されなかった理由をご教示ください。
        *   測定が実施されなかった場合、評価スケジュールからの逸脱となります。データ欠損の理由と、今後のデータ収集について、ご計画があればご教示ください。
    *   **判断理由:** バイタルサイン測定の未実施は、プロトコルで規定された評価項目の一部欠落であり、データの完全性に影響を与える可能性があるため。